# 03 - Retrieval-depth evaluation

## Purpose

This notebook records the retrieval-only evaluation used to select the baseline
retrieval depth `k`. It uses the fixed 20-question benchmark, the validated
production embeddings and the exact FAISS index.

The notebook has three deliberately separate actions:

1. one paid request embeds the 20 questions and prepares a rank-blinded pool;
2. the candidate pairs are judged locally using grades 0, 1 or 2;
3. the completed judgements are scored offline using the frozen selection rule.

This separation supports the dissertation assessment requirements for a clear
method, reproducible evidence, critical justification of design choices and
responsible handling of evaluation data.


## Evaluation and leakage boundary

- This is retrieval evaluation, not answer generation.
- The 20 questions are embedded only as runtime queries and are never added to
  the corpus index.
- Ground-truth answers are not loaded or used at any point in this notebook.
- The judgement view displays only a question and one candidate chunk. It hides
  ranks, similarity scores, question identifiers, chunk identifiers and page
  provenance.
- Candidate depths are evaluated from one frozen depth-20 search. This avoids
  changing the retrieved evidence after relevance judgements have started.
- The same 20 questions are used for calibration and later model evaluation, so
  the selected `k` is a benchmark-tuned setting rather than an independent
  holdout estimate. This limitation must be reported.

The paid, annotation and scoring cells are disabled by default. Therefore,
running all cells from a fresh kernel does not make an API request or change an
evaluation file.


In [1]:
from __future__ import annotations

from html import escape
from pathlib import Path
from tempfile import TemporaryDirectory
import json
import os
import sys

from IPython.display import HTML, clear_output, display
import numpy as np


def locate_project_root() -> Path:
    """Locate the repository from its root or notebooks directory."""

    candidates = [Path.cwd().resolve(), Path.cwd().resolve().parent]
    for candidate in candidates:
        if (
            (candidate / "pyproject.toml").is_file()
            and (candidate / "configs/retrieval-evaluation-config.json").is_file()
        ):
            return candidate
    raise RuntimeError(
        "Run this notebook from the repository root or its notebooks directory."
    )


def resolve_project_path(root: Path, relative_path: str) -> Path:
    """Resolve a configured path without allowing it to escape the project."""

    path = (root / relative_path).resolve()
    if not path.is_relative_to(root):
        raise RuntimeError(f"Configured path leaves the project: {relative_path}")
    return path


def env_file_defines_key(path: Path, key: str) -> bool:
    """Check whether a key name exists without reading or printing its value."""

    if not path.is_file():
        return False
    for raw_line in path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        name = line.split("=", 1)[0].strip()
        if name.startswith("export "):
            name = name.removeprefix("export ").strip()
        if name == key:
            return True
    return False


PROJECT_ROOT = locate_project_root()
SOURCE_ROOT = (PROJECT_ROOT / "src").resolve()
if not SOURCE_ROOT.is_dir():
    raise RuntimeError(f"Project source directory is missing: {SOURCE_ROOT}")

source_root_string = str(SOURCE_ROOT)
if source_root_string not in sys.path:
    sys.path.insert(0, source_root_string)

from geotech_rag.retrieval_evaluation import (
    calculate_file_sha256,
    load_retrieval_evaluation_config,
    load_retrieval_evaluation_inputs,
    prepare_retrieval_evaluation,
    score_retrieval_judgements,
    serialise_json_lines,
)


CONFIG_PATH = PROJECT_ROOT / "configs/retrieval-evaluation-config.json"
NOTEBOOK_PATH = PROJECT_ROOT / "notebooks/03_retrieval_depth_evaluation.ipynb"

print("Project root:", PROJECT_ROOT)
print("Source root added to Python path:", SOURCE_ROOT)
print("Configuration:", CONFIG_PATH.relative_to(PROJECT_ROOT))
print("Ground truth accessed: False")


Project root: /home/zaki/coding_F/GismaProjects/Thesis
Source root added to Python path: /home/zaki/coding_F/GismaProjects/Thesis/src
Configuration: configs/retrieval-evaluation-config.json
Ground truth accessed: False


## 1. Frozen protocol and input preflight

This preflight loads the committed protocol and validates every referenced
question, corpus, embedding, mapping and FAISS fingerprint. It performs no API
request and does not display question or chunk text.


In [2]:
config = load_retrieval_evaluation_config(CONFIG_PATH)
inputs = load_retrieval_evaluation_inputs(CONFIG_PATH, PROJECT_ROOT)
outputs = config["outputs"]

QUERY_MATRIX_PATH = resolve_project_path(
    PROJECT_ROOT,
    outputs["query_embedding_matrix_relative_path"],
)
CANDIDATE_POOL_PATH = resolve_project_path(
    PROJECT_ROOT,
    outputs["ranked_candidate_pool_relative_path"],
)
JUDGEMENT_PATH = resolve_project_path(
    PROJECT_ROOT,
    outputs["blinded_judgement_file_relative_path"],
)
SUMMARY_PATH = resolve_project_path(
    PROJECT_ROOT,
    outputs["summary_relative_path"],
)
METRICS_PATH = resolve_project_path(
    PROJECT_ROOT,
    outputs["metrics_relative_path"],
)
METRICS_TABLE_PATH = resolve_project_path(
    PROJECT_ROOT,
    outputs["metrics_table_relative_path"],
)

question_count = len(inputs.question_records)
pool_depth = config["search"]["judgement_pool_depth"]
expected_candidate_count = question_count * pool_depth

print("Retrieval-evaluation preflight:")
print("  Configuration ID:", config["configuration_id"])
print("  Questions:", question_count)
print("  Corpus chunks:", len(inputs.chunk_records))
print("  FAISS vectors:", inputs.faiss_index.ntotal)
print("  Candidate depths:", config["search"]["candidate_depths"])
print("  LangChain reference depth:", config["search"]["reference_depth"])
print("  Judgement pool depth:", pool_depth)
print("  Expected candidate pairs:", expected_candidate_count)
print("  Selection status:", config["selection_rule"]["selection_status"])
print("  OPENAI_API_KEY exported:", bool(os.getenv("OPENAI_API_KEY")))
print(
    "  OPENAI_API_KEY named in .env:",
    env_file_defines_key(PROJECT_ROOT / ".env", "OPENAI_API_KEY"),
)
print("  Credential value displayed: False")
print("  Ground truth accessed: False")
print("Retrieval-evaluation preflight: PASSED")


Retrieval-evaluation preflight:
  Configuration ID: benchmark-retrieval-depth-evaluation-v1
  Questions: 20
  Corpus chunks: 5113
  FAISS vectors: 5113
  Candidate depths: [1, 2, 3, 4, 5, 8, 10]
  LangChain reference depth: 4
  Judgement pool depth: 20
  Expected candidate pairs: 400
  Selection status: unresolved_pending_relevance_judgements
  OPENAI_API_KEY exported: True
  OPENAI_API_KEY named in .env: True
  Credential value displayed: False
  Ground truth accessed: False
Retrieval-evaluation preflight: PASSED


## 2. Guarded paid preparation

The preparation function sends the 20 ordered question texts through the frozen
embedding wrapper in one request. It then L2-normalises the returned float32
vectors, searches the existing exact FAISS index once at depth 20, and writes:

- the query embedding matrix;
- the private ranked candidate pool;
- the private rank-blinded judgement template;
- a text-free preparation summary.

Leave `RUN_PAID_PREPARATION = False` while reviewing the notebook. Change it to
`True` only once, run this cell, and then restore it to `False`. Do not use
overwrite for the initial production run.


In [ ]:
RUN_PAID_PREPARATION = False

if not RUN_PAID_PREPARATION:
    print("Paid retrieval preparation: NOT RUN")
    print("Set RUN_PAID_PREPARATION = True only for the initial production run.")
else:
    preparation_result = prepare_retrieval_evaluation(
        CONFIG_PATH,
        PROJECT_ROOT,
        overwrite=False,
    )
    print("Paid retrieval preparation: PASSED")
    print("  Questions:", preparation_result.question_count)
    print("  Judgement pool depth:", preparation_result.pool_depth)
    print("  Candidate pairs:", preparation_result.candidate_count)
    print(
        "  Query embedding matrix:",
        preparation_result.query_embedding_matrix_path.relative_to(PROJECT_ROOT),
    )
    print("  Query matrix SHA-256:", preparation_result.query_embedding_matrix_sha256)
    print(
        "  Ranked candidate pool:",
        preparation_result.ranked_candidate_pool_path.relative_to(PROJECT_ROOT),
    )
    print("  Candidate pool SHA-256:", preparation_result.ranked_candidate_pool_sha256)
    print(
        "  Blinded judgement file:",
        preparation_result.blinded_judgement_file_path.relative_to(PROJECT_ROOT),
    )
    print(
        "  Judgement template SHA-256:",
        preparation_result.blinded_judgement_file_sha256,
    )
    print("  Summary:", preparation_result.summary_path.relative_to(PROJECT_ROOT))
    print("  Summary SHA-256:", preparation_result.summary_sha256)
    print("  Ground truth accessed: False")
    print("  Selected retrieval depth k: unresolved")


Paid retrieval preparation: PASSED
  Questions: 20
  Judgement pool depth: 20
  Candidate pairs: 400
  Query embedding matrix: data/processed/evaluation/retrieval/query-embeddings.npy
  Query matrix SHA-256: fddd63335e250d875781890cec48cc646165e221d9b0cabdd22f489dc35be882
  Ranked candidate pool: data/processed/evaluation/retrieval/ranked-candidate-pool.jsonl
  Candidate pool SHA-256: 1713a14a25433abde59791eede325f33605da86da85019e2cda3597ec0ffc114
  Blinded judgement file: data/raw/evaluation/retrieval-judgements/relevance-judgements.jsonl
  Judgement template SHA-256: bbc72d5e423bf5d745ec93cb9b6de952520135039a3f6dac5132b3a784b718e1
  Summary: data/processed/audit/retrieval-depth-evaluation-summary.json
  Summary SHA-256: 6083b0048d45905cd6724b472a6ec6f577a16f5c98984e9bde4fa8b01402e040
  Ground truth accessed: False
  Selected retrieval depth k: unresolved


## 3. Preparation artifact audit

Run this cell after the guarded preparation has passed. It checks shapes,
counts, fingerprints and the blinding schema. It deliberately does not print
any query text, chunk text, rank, score or provenance.


In [4]:
preparation_paths = [
    QUERY_MATRIX_PATH,
    CANDIDATE_POOL_PATH,
    JUDGEMENT_PATH,
    SUMMARY_PATH,
]

if not all(path.is_file() for path in preparation_paths):
    print("Preparation artifact audit: NOT RUN")
    print("Run the guarded paid preparation first.")
else:
    summary = json.loads(SUMMARY_PATH.read_text(encoding="utf-8"))
    query_matrix = np.load(QUERY_MATRIX_PATH, allow_pickle=False)
    with CANDIDATE_POOL_PATH.open("r", encoding="utf-8") as handle:
        candidate_records = [json.loads(line) for line in handle if line.strip()]
    with JUDGEMENT_PATH.open("r", encoding="utf-8") as handle:
        judgement_records = [json.loads(line) for line in handle if line.strip()]

    expected_judgement_fields = {
        "chunk_text",
        "configuration_id",
        "judgement_id",
        "judgement_note",
        "judgement_schema_version",
        "query_text",
        "relevance_grade",
    }
    hidden_judgement_fields = {
        "question_id",
        "chunk_id",
        "question_position",
        "index_position",
        "rank",
        "similarity_score",
        "parent_record_id",
        "source_id",
        "pdf_page_index",
        "pdf_page_number",
        "printed_page_number",
    }

    if query_matrix.shape != (
        question_count,
        config["query_embedding"]["dimensions"],
    ):
        raise RuntimeError(f"Unexpected query matrix shape: {query_matrix.shape}")
    if query_matrix.dtype != np.float32 or not query_matrix.flags.c_contiguous:
        raise RuntimeError("Query matrix does not follow the float32 C-contiguous contract.")
    if not np.isfinite(query_matrix).all():
        raise RuntimeError("Query matrix contains a non-finite value.")
    if len(candidate_records) != expected_candidate_count:
        raise RuntimeError("Candidate count differs from the frozen protocol.")
    if len(judgement_records) != expected_candidate_count:
        raise RuntimeError("Judgement count differs from the frozen protocol.")
    if any(set(record) != expected_judgement_fields for record in judgement_records):
        raise RuntimeError("A blinded judgement record has unexpected fields.")
    if any(hidden_judgement_fields.intersection(record) for record in judgement_records):
        raise RuntimeError("A rank or provenance field leaked into the judgement file.")
    if summary["question_embedding_matrix_sha256"] != calculate_file_sha256(
        QUERY_MATRIX_PATH
    ):
        raise RuntimeError("Query matrix fingerprint differs from the summary.")
    if summary["ranked_candidate_pool_sha256"] != calculate_file_sha256(
        CANDIDATE_POOL_PATH
    ):
        raise RuntimeError("Candidate-pool fingerprint differs from the summary.")

    query_norms = np.linalg.norm(query_matrix, axis=1)
    completed_count = sum(
        record["relevance_grade"] is not None for record in judgement_records
    )

    print("Preparation artifact audit:")
    print("  Query matrix shape:", query_matrix.shape)
    print("  Query matrix dtype:", query_matrix.dtype)
    print("  Query matrix C-contiguous:", query_matrix.flags.c_contiguous)
    print("  Finite values:", bool(np.isfinite(query_matrix).all()))
    print("  Minimum norm:", float(query_norms.min()))
    print("  Maximum norm:", float(query_norms.max()))
    print("  Candidate records:", len(candidate_records))
    print("  Blinded judgement records:", len(judgement_records))
    print("  Completed judgements:", completed_count)
    print("  Hidden rank, score and provenance fields present: False")
    print("  Question or chunk text displayed: False")
    print("  Ground truth accessed: False")
    print("Preparation artifact audit: PASSED")


Preparation artifact audit:
  Query matrix shape: (20, 1536)
  Query matrix dtype: float32
  Query matrix C-contiguous: True
  Finite values: True
  Minimum norm: 0.9999998211860657
  Maximum norm: 1.000000238418579
  Candidate records: 400
  Blinded judgement records: 400
  Completed judgements: 0
  Hidden rank, score and provenance fields present: False
  Question or chunk text displayed: False
  Ground truth accessed: False
Preparation artifact audit: PASSED


## 4. Blinded relevance judgement helper

Use the following grades consistently:

- `0` - irrelevant: the chunk does not help answer the question;
- `1` - supporting: the chunk gives related context, a definition, or a useful
  intermediate fact, but it does not directly provide the main evidence;
- `2` - direct: the chunk directly contains the relationship, procedure, data,
  equation or explanation needed for the question.

Grades 1 and 2 require a short note explaining the evidence. The helper writes
the whole JSONL file atomically after every judgement, so completed work is not
lost if the session stops later.

The question and chunk are shown only while an annotation prompt is active.
The output is cleared when the session ends, including after an interruption.
We should not save or commit the notebook while private text is visible on screen.


In [5]:
EXPECTED_JUDGEMENT_FIELDS = {
    "chunk_text",
    "configuration_id",
    "judgement_id",
    "judgement_note",
    "judgement_schema_version",
    "query_text",
    "relevance_grade",
}


def load_local_judgements() -> list[dict[str, object]]:
    """Load the private blinded file without displaying its text."""

    if not JUDGEMENT_PATH.is_file():
        raise RuntimeError("The blinded judgement file does not exist yet.")
    with JUDGEMENT_PATH.open("r", encoding="utf-8") as handle:
        records = [json.loads(line) for line in handle if line.strip()]
    if len(records) != expected_candidate_count:
        raise RuntimeError("Unexpected number of blinded judgement records.")
    if any(set(record) != EXPECTED_JUDGEMENT_FIELDS for record in records):
        raise RuntimeError("A blinded judgement record has unexpected fields.")
    if len({record["judgement_id"] for record in records}) != len(records):
        raise RuntimeError("Blinded judgement identifiers are not unique.")
    return records


def save_local_judgements(records: list[dict[str, object]]) -> None:
    """Atomically replace the private judgement file after validation."""

    if len(records) != expected_candidate_count:
        raise RuntimeError("Refusing to save an incomplete judgement dataset.")
    if any(set(record) != EXPECTED_JUDGEMENT_FIELDS for record in records):
        raise RuntimeError("Refusing to save a record with unexpected fields.")

    JUDGEMENT_PATH.parent.mkdir(parents=True, exist_ok=True)
    with TemporaryDirectory(dir=JUDGEMENT_PATH.parent) as temporary_directory:
        temporary_path = Path(temporary_directory) / JUDGEMENT_PATH.name
        temporary_path.write_bytes(serialise_json_lines(records))
        with temporary_path.open("r", encoding="utf-8") as handle:
            reloaded = [json.loads(line) for line in handle if line.strip()]
        if reloaded != records:
            raise RuntimeError("Judgements changed during the save round trip.")
        os.replace(temporary_path, JUDGEMENT_PATH)


def judgement_progress() -> tuple[int, int]:
    """Return completed and total counts without exposing private content."""

    records = load_local_judgements()
    completed = sum(record["relevance_grade"] is not None for record in records)
    return completed, len(records)


def annotation_session(max_items: int = 10) -> None:
    """Judge up to max_items private pairs and save after every response."""

    if isinstance(max_items, bool) or not isinstance(max_items, int) or max_items < 1:
        raise ValueError("max_items must be a positive integer.")

    processed = 0
    try:
        while processed < max_items:
            records = load_local_judgements()
            next_position = next(
                (
                    position
                    for position, record in enumerate(records)
                    if record["relevance_grade"] is None
                ),
                None,
            )
            if next_position is None:
                break

            record = records[next_position]
            completed = sum(
                value["relevance_grade"] is not None for value in records
            )
            clear_output(wait=True)
            display(
                HTML(
                    "<h3>Blinded retrieval judgement</h3>"
                    f"<p>Progress before save: {completed} / {len(records)}</p>"
                    "<h4>Question</h4>"
                    f"<pre style='white-space:pre-wrap'>{escape(str(record['query_text']))}</pre>"
                    "<h4>Candidate chunk</h4>"
                    f"<pre style='white-space:pre-wrap'>{escape(str(record['chunk_text']))}</pre>"
                    "<p><strong>0</strong> irrelevant, "
                    "<strong>1</strong> supporting, "
                    "<strong>2</strong> direct</p>"
                )
            )

            raw_grade = input("Grade 0, 1, 2, or q to stop: ").strip().lower()
            if raw_grade == "q":
                break
            if raw_grade not in {"0", "1", "2"}:
                print("Invalid grade. This pair was not changed.")
                input("Press Enter to continue: ")
                continue

            grade = int(raw_grade)
            note: str | None = None
            if grade in {1, 2}:
                note = input("Short evidence note: ").strip()
                if not note:
                    print("Grades 1 and 2 require a note. This pair was not changed.")
                    input("Press Enter to continue: ")
                    continue

            record["relevance_grade"] = grade
            record["judgement_note"] = note
            save_local_judgements(records)
            processed += 1
    finally:
        clear_output(wait=False)
        if JUDGEMENT_PATH.is_file():
            completed, total = judgement_progress()
            print("Annotation session closed safely.")
            print("  Judgements saved in this session:", processed)
            print("  Completed judgements:", completed)
            print("  Remaining judgements:", total - completed)
            print("  Total judgements:", total)
            print("  Private question or chunk text retained in cell output: False")


print("Blinded annotation helper: READY")
print("No judgement was changed by defining these functions.")


Blinded annotation helper: READY
No judgement was changed by defining these functions.


## 5. Guarded annotation session

Change `RUN_ANNOTATION_SESSION` to `True` and run this cell when ready to judge
a small block. Ten pairs per session gives regular stopping points and keeps the
work manageable. Enter `q` to stop early. Restore the flag to `False` after the
session.

Do not consult the ranked candidate pool, similarity scores, source pages,
question IDs, ground truth or model-generated answers while judging.


In [ ]:
RUN_ANNOTATION_SESSION = False
ANNOTATION_BLOCK_SIZE = 10

if not RUN_ANNOTATION_SESSION:
    print("Blinded annotation session: NOT RUN")
    if JUDGEMENT_PATH.is_file():
        completed, total = judgement_progress()
        print("  Completed judgements:", completed)
        print("  Remaining judgements:", total - completed)
        print("  Total judgements:", total)
else:
    annotation_session(max_items=ANNOTATION_BLOCK_SIZE)


Annotation session closed safely.
  Judgements saved in this session: 10
  Completed judgements: 400
  Remaining judgements: 0
  Total judgements: 400
  Private question or chunk text retained in cell output: False


## 6. Completion audit

This audit reports only counts and the current private judgement-file
fingerprint. The fingerprint is expected to change after each saved annotation.
It does not display any private content.


In [48]:
if not JUDGEMENT_PATH.is_file():
    print("Completion audit: NOT RUN")
    print("The blinded judgement file does not exist yet.")
else:
    completed, total = judgement_progress()
    print("Blinded judgement completion audit:")
    print("  Completed judgements:", completed)
    print("  Remaining judgements:", total - completed)
    print("  Total judgements:", total)
    print("  Current judgement SHA-256:", calculate_file_sha256(JUDGEMENT_PATH))
    print("  Question or chunk text displayed: False")
    print("  Ground truth accessed: False")
    if completed == total:
        print("Blinded judgement completion audit: PASSED")
    else:
        print("Blinded judgement completion audit: INCOMPLETE")


Blinded judgement completion audit:
  Completed judgements: 400
  Remaining judgements: 0
  Total judgements: 400
  Current judgement SHA-256: 33f408a2f744d662f78d9eaa6c77d9a1f321d487899f7a2758bf162646ca8fe9
  Question or chunk text displayed: False
  Ground truth accessed: False
Blinded judgement completion audit: PASSED


## 7. Guarded offline scoring

Scoring is allowed only after all 400 judgements are complete. It validates the
private candidate and judgement files, calculates every candidate depth from the
same frozen ranking, applies the adequacy gate and then applies the frozen
selection rule:

1. maximise the number of questions with direct evidence;
2. if tied, maximise the number with useful evidence;
3. if still tied, select the smallest `k`.

This step makes no API request and does not load ground-truth answers. Leave the
flag disabled until the completion audit passes.


In [8]:
RUN_OFFLINE_SCORING = False

if not RUN_OFFLINE_SCORING:
    print("Offline retrieval scoring: NOT RUN")
    print("Set RUN_OFFLINE_SCORING = True only after all judgements are complete.")
else:
    completed, total = judgement_progress()
    if completed != total:
        raise RuntimeError(
            f"Cannot score incomplete judgements: {completed} / {total} complete."
        )
    metrics_result = score_retrieval_judgements(
        CONFIG_PATH,
        PROJECT_ROOT,
        overwrite=False,
    )
    print("Offline retrieval scoring: PASSED")
    print("  Completed judgements:", metrics_result.completed_judgement_count)
    print(
        "  Candidate-range adequacy gate triggered:",
        metrics_result.candidate_range_adequacy_gate_triggered,
    )
    print("  Selected retrieval depth k:", metrics_result.selected_depth_k)
    print("  Metrics:", metrics_result.metrics_path.relative_to(PROJECT_ROOT))
    print("  Metrics SHA-256:", metrics_result.metrics_sha256)
    print("  Metrics table:", metrics_result.metrics_table_path.relative_to(PROJECT_ROOT))
    print("  Metrics table SHA-256:", metrics_result.metrics_table_sha256)
    print("  API request made during scoring: False")
    print("  Ground truth accessed: False")


Offline retrieval scoring: NOT RUN
Set RUN_OFFLINE_SCORING = True only after all judgements are complete.


## 8. Text-free metric review

After scoring, this cell shows the frozen metric rows and selection result. The
tracked results contain aggregated numbers and fingerprints only, with no
question text or chunk text.


In [9]:
if not METRICS_PATH.is_file() or not METRICS_TABLE_PATH.is_file():
    print("Metric review: NOT RUN")
    print("Run guarded offline scoring after completing every judgement.")
else:
    metrics = json.loads(METRICS_PATH.read_text(encoding="utf-8"))
    metrics_text = METRICS_PATH.read_text(encoding="utf-8")

    if "query_text" in metrics_text or "chunk_text" in metrics_text:
        raise RuntimeError("Tracked metrics contain a forbidden text field name.")

    print("Retrieval-depth metric review:")
    print("  Selected retrieval depth k:", metrics["selected_depth_k"])
    print("  Selection status:", metrics["selection_status"])
    print(
        "  Candidate-range adequacy gate triggered:",
        metrics["candidate_range_adequacy_gate"]["triggered"],
    )
    print("  Completed judgements:", metrics["completed_judgement_count"])
    print("  Metrics SHA-256:", calculate_file_sha256(METRICS_PATH))
    print("  Metrics table SHA-256:", calculate_file_sha256(METRICS_TABLE_PATH))
    print("  Question or chunk text displayed: False")
    print("  Ground truth accessed: False")
    print()
    print(METRICS_TABLE_PATH.read_text(encoding="utf-8"))
    print("Retrieval-depth metric review: PASSED")


Metric review: NOT RUN
Run guarded offline scoring after completing every judgement.


## 9. Reproducibility and reporting boundary

The retrieval depth is selected from the frozen candidate set only after blinded
manual relevance judgement. It must remain unresolved if the adequacy gate finds
direct evidence at ranks 11-20 for any question with no direct evidence at ranks
1-10. In that case, the protocol must be versioned and expanded before a depth is
selected.

The final dissertation should report the selected depth together with the full
candidate table, the adequacy-gate result, the judgement rules and the limitation
that the same 20 benchmark questions were used for calibration. Retrieval
evaluation does not claim whole-corpus recall because the corpus was not
exhaustively judged.

Before committing this notebook, restore all three action flags to `False`, save
the notebook, validate its JSON and confirm that no annotation cell output
contains a question or chunk text. Private preparation and judgement artifacts
remain ignored by Git. Only the final text-free metrics and the documented audit
may be tracked later.
